### Load data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wandb/RAGTruth-processed")

Repo card metadata block was not found. Setting CardData to empty.


In [18]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 15090
    })
    test: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 2700
    })
})


In [19]:
import pandas as pd

# Convert the 'train' split of the dataset to a pandas DataFrame
df = pd.DataFrame(dataset['train'])

In [20]:
# Show the first 5 rows of both 'hallucination_labels' and 'hallucination_labels_processed' columns
display(df[['hallucination_labels', 'hallucination_labels_processed']].head())

,hallucination_labels,hallucination_labels_processed
0,[],"{'evident_conflict': 0, 'baseless_info': 0}"
1,[],"{'evident_conflict': 0, 'baseless_info': 0}"
2,"[{""start"": 636, ""end"": 653, ""text"": ""February ...","{'evident_conflict': 1, 'baseless_info': 1}"
3,"[{""start"": 693, ""end"": 754, ""text"": ""Anne's di...","{'evident_conflict': 0, 'baseless_info': 1}"
4,[],"{'evident_conflict': 0, 'baseless_info': 0}"


In [22]:
type(df.loc[2, "hallucination_labels_processed"])

dict

### Label

In [30]:
def to_label(processed: dict[str, int]) -> int:
    return 0 if all(v == 0 for v in processed.values()) else 1

df["label"] = df["hallucination_labels_processed"].apply(to_label)

### Class balance

In [31]:
print(df["label"].value_counts())

label
0    8369
1    6721
Name: count, dtype: int64


### Per task type

In [25]:
df.groupby("task_type")["label"].mean()

,label
task_type,
Data2txt,0.693658
QA,0.310687
Summary,0.311475


In [26]:
df.groupby("task_type")["label"].count()

,label
task_type,
Data2txt,5298
QA,5034
Summary,4758


### Lengths

In [27]:
df["context_words"] = df["context"].str.split().str.len()
df["output_words"] = df["output"].str.split().str.len()
df[["context_words", "output_words"]].describe()

,context_words,output_words
count,15090.000000,15090.000000
mean,377.743141,131.363685
std,245.350918,57.214785
min,72.000000,1.000000
25%,224.000000,92.000000
50%,317.000000,125.000000
75%,426.000000,163.000000
max,1750.000000,632.000000


In [28]:
df.groupby("label")["output_words"].mean()

,output_words
label,
0,119.319393
1,146.361256


In [29]:
df.groupby("label")["context_words"].mean()

,context_words
label,
0,380.398016
1,374.437286
